# CDR-MLC Cluster-Conditioned Residualization

This diagnostic experiment estimates the congestion-dependent component of the 32 classification features from the causal 15-dimensional routing vector. A separate multi-output Ridge mapper is fitted inside each training cluster.

Each expert receives the original features, the residual-modulated features, and the three soft congestion memberships. Test labels are used only after prediction for evaluation. The fixed residual strength is 0.5; it is not selected from test performance.

This is an experimental ablation and does not replace the baseline CDR-MLC notebook.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MAIN = Path("CDR-MLC.ipynb")
if not MAIN.exists():
    MAIN = Path("CDR_MLC") / "CDR-MLC.ipynb"
namespace = {}
with MAIN.open(encoding="utf-8") as handle:
    main_notebook = json.load(handle)
exec(compile("".join(main_notebook["cells"][0]["source"]), str(MAIN), "exec"), namespace)
run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]
compute_sliding_window_stats = namespace["compute_sliding_window_stats"]

try:
    display
except NameError:
    display = print


def _soft_membership(distances):
    inverse = 1.0 / np.maximum(distances, 1e-9) ** 2
    return inverse / inverse.sum(axis=1, keepdims=True)


def run_cluster_residualization(train_file, test_file, residual_strength=0.5):
    baseline = run_pipeline_from_two_files(
        train_file, test_file, n_clusters=3, window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )
    train, test = baseline["train_df"], baseline["test_df"]
    features = baseline["classification_features"]
    target = baseline["target_column"]
    X_train = train[features].to_numpy(float)
    X_test = test[features].to_numpy(float)
    y_train = train[target].to_numpy()
    train_routes = train["cluster"].to_numpy()
    test_routes = test["cluster"].to_numpy()

    train_stats, _ = compute_sliding_window_stats(
        train[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    test_stats, _ = compute_sliding_window_stats(
        test[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    G_train = baseline["scaler"].transform(train_stats)
    G_test = baseline["scaler"].transform(test_stats)
    M_train = _soft_membership(
        baseline["kmeans_model"].transform(G_train))
    M_test = _soft_membership(
        baseline["kmeans_model"].transform(G_test))

    experts, residualizers = {}, {}
    for cluster_id in range(3):
        mask = train_routes == cluster_id
        mapper = Ridge(alpha=1.0)
        mapper.fit(G_train[mask], X_train[mask])
        reference = mapper.predict(G_train[mask]).mean(axis=0)
        adjusted = X_train[mask] - residual_strength * (
            mapper.predict(G_train[mask]) - reference)
        expert_input = np.hstack([
            X_train[mask], adjusted, M_train[mask]])
        expert = RandomForestClassifier(
            n_estimators=80, random_state=42,
            class_weight="balanced", n_jobs=-1)
        expert.fit(expert_input, y_train[mask])
        residualizers[cluster_id] = (mapper, reference)
        experts[cluster_id] = expert

    predictions = np.empty(len(X_test), dtype=int)
    for cluster_id in range(3):
        mask = test_routes == cluster_id
        if not mask.any():
            continue
        mapper, reference = residualizers[cluster_id]
        adjusted = X_test[mask] - residual_strength * (
            mapper.predict(G_test[mask]) - reference)
        expert_input = np.hstack([
            X_test[mask], adjusted, M_test[mask]])
        predictions[mask] = experts[cluster_id].predict(expert_input)

    # Evaluation boundary.
    y_test = test[target].to_numpy()
    residual_metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(
            y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(
            y_test, predictions, average="weighted"),
        "f1_weighted": f1_score(
            y_test, predictions, average="weighted"),
        "f1_macro": f1_score(y_test, predictions, average="macro"),
    }
    names = list(residual_metrics)
    comparison = pd.DataFrame([
        {"method": "hard_cdr_mlc", **{
            name: baseline["test_results"][name] for name in names}},
        {"method": "cluster_residualization", **residual_metrics},
    ])
    for name in names:
        base = comparison.loc[
            comparison["method"] == "hard_cdr_mlc", name].iloc[0]
        comparison[f"{name}_gain_pp"] = 100 * (
            comparison[name] - base)
    display(comparison.round(4))
    return {
        "baseline": baseline,
        "experts": experts,
        "residualizers": residualizers,
        "predictions": predictions,
        "comparison": comparison,
    }


In [ ]:
# scenario_1: run independently
scenario_1_residual = run_cluster_residualization(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
)


In [ ]:
# scenario_2: run independently
scenario_2_residual = run_cluster_residualization(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_3: run independently
scenario_3_residual = run_cluster_residualization(
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_4: run independently
scenario_4_residual = run_cluster_residualization(
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
)


In [ ]:
# scenario_5: run independently
scenario_5_residual = run_cluster_residualization(
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
)
